# 🏥 [KTAS-LLM 어블레이션] Arm 2. Tree-Only (의학 룰 트리만) LoRA 파인튜닝 & 학습 모니터링
> **실험군 목표 (Arm 2)**: 일상 환자 구어체 발화 없이, **공식 KTAS 임상 규칙 지침문 $\rightarrow$ 적용 룰 ID 및 KTAS 단계** 매핑만 학습한 '의학 법전 암기형' 모델을 구축합니다.  
> **핵심 과학적 질문**: 구어체 잡음 없이 완벽한 결정 트리 규칙만 학습한 모델은 공식 룰 암기력은 높지만, 실제 환자의 비정형 구어체와 순서 편향을 만났을 때 일반화 성능이 어떻게 붕괴(Degrade)하는가?

---

### 🗺️ 본 노트북의 워크플로우 (Workflow Roadmap)
| 단계 | 작업 내용 | 주요 기술 및 엔지니어링 |
| :---: | :--- | :--- |
| **Step 1** | **하드웨어 및 PyTorch 가속 환경 진단** | GB10 128GB Unified Memory, CUDA 13.2, SDPA 점검 |
| **Step 2** | **Arm 2 데이터셋 로드 & ChatML 토큰화** | `arm2_train_tree_only.jsonl` (3,240건) 토큰 길이 분포 시각화 |
| **Step 3** | **Qwen2.5-32B 원본 동결 & LoRA 어댑터 바인딩** | 62GB 원본 가중치 동결 + r=16, alpha=32 어댑터 구성 |
| **Step 4** | **실시간 학습 곡선 모니터링 콜백 정의** | 스텝별 Train/Val Loss 및 Learning Rate 실시간 인터랙티브 플롯 |
| **Step 5** | **표준 Trainer 기반 파인튜닝 가동** | 배치 16 (1 x 16), bf16, Gradient Checkpointing 안정 학습 |
| **Step 6** | **최종 학습 수렴도 요약 및 어댑터 저장** | `outputs/qwen2.5-32b-arm2-tree-only` 어댑터 저장 및 PPL 시각화 |


---
## Step 1. 하드웨어 인프라 및 PyTorch 가속 환경 점검
### 💡 왜 이 작업을 하나요?
* **NVIDIA GB10 (128GB Unified Memory)** 환경에서 동일한 32B 모델 가중치에 대해 Arm 2 전용 LoRA 파인튜닝 환경을 초기화합니다.


In [ ]:
# Step 1 코드: 하드웨어 및 실행 환경 점검
import os
import sys
import json
import time
import math
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, clear_output

# 프로젝트 및 실험 디렉토리 경로 자동 감지
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "models").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR.parent.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent.parent
else:
    PROJECT_ROOT = Path("/home/meisteryi/문서/KTAS-LLM")

EXP_DIR = PROJECT_ROOT / "experiments" / "02_tree_ablation_xai"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = EXP_DIR / "data"
OUTPUTS_DIR = EXP_DIR / "outputs"
REPORTS_DIR = EXP_DIR / "reports"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# 시각화 기본 스타일
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

print("=" * 70)
print("[Step 1] Arm 2 (Tree-Only) 실행 환경 진단")
print(f" - 프로젝트 루트 : {PROJECT_ROOT}")
print(f" - 실험 디렉토리 : {EXP_DIR}")
print(f" - 모델 디렉토리 : {MODELS_DIR / 'Qwen2.5-32B-Instruct'}")
print(f" - Python 버전   : {sys.version.split()[0]}")
print(f" - PyTorch 버전  : {torch.__version__}")
print(f" - CUDA 지원     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f" - 가속기(GPU)   : {device_name}")
    print(f" - 가용 통합메모리: {total_mem:.2f} GB (충분함)")
else:
    print(" [!] 경고: GPU가 감지되지 않았습니다.")
print("=" * 70)


---
## Step 2. Arm 2 (Tree-Only) 데이터셋 로드 & ChatML 토큰화
### 💡 Arm 2 데이터의 구조적 특징
* **입력(User)**: 환자 발화가 아닌, 딱딱한 임상 지침 시나리오 문장:
  `"[임상 규칙 시나리오]: "급성 호흡곤란 및 흉통 증상이 관찰되며, 돌발적인 호흡곤란 및 객혈... KTAS 2단계 긴급에 해당함.""`
* **출력(Assistant)**: 룰 ID와 최종 단계:
  ```json
  {
    "applied_rule_id": "호흡기계_103_2차고려_급성중등도_호흡곤란",
    "ktas_level": 2
  }
  ```
* 환자의 사소한 잡음이나 구어체가 완전히 배제된 이상적인 규칙 상태만을 학습합니다.


In [ ]:
# Step 2 코드: Arm 2 데이터 로드 및 토큰화
from transformers import AutoTokenizer
from datasets import Dataset

model_dir = str(MODELS_DIR / "Qwen2.5-32B-Instruct")
train_path = str(DATA_DIR / "arm2_train_tree_only.jsonl")
val_path = str(DATA_DIR / "arm2_val_tree_only.jsonl")

print(f"[*] 모델 경로 확인: {model_dir}")
assert os.path.exists(model_dir), f"모델 폴더 없음: {model_dir}"
assert os.path.exists(train_path), f"학습 데이터 없음: {train_path}"

print("[*] Qwen2.5 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_samples = load_jsonl(train_path)
val_samples = load_jsonl(val_path)

def preprocess_function(sample):
    messages = sample["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, max_length=512, truncation=True)
    enc["labels"] = enc["input_ids"].copy()
    return enc

print("[*] ChatML 템플릿 토큰화 진행 중...")
raw_train_ds = Dataset.from_list(train_samples)
raw_val_ds = Dataset.from_list(val_samples)

train_dataset = raw_train_ds.map(preprocess_function, remove_columns=["messages"], batched=False)
val_dataset = raw_val_ds.map(preprocess_function, remove_columns=["messages"], batched=False)

print(f"[+] 토큰화 완료: Train {len(train_dataset):,} 건, Val {len(val_dataset):,} 건")

# 토큰 길이 분포 시각화
token_lengths = [len(x["input_ids"]) for x in train_dataset]
max_len = max(token_lengths)
mean_len = sum(token_lengths) / len(token_lengths)

plt.figure(figsize=(10, 4), dpi=130)
sns.histplot(token_lengths, bins=35, kde=True, color="#8e44ad", edgecolor="black")
plt.axvline(x=max_len, color="#e74c3c", linestyle="--", lw=1.5, label=f"Max Length ({max_len} toks)")
plt.axvline(x=mean_len, color="#f39c12", linestyle="-", lw=1.5, label=f"Mean Length ({mean_len:.1f} toks)")
plt.title("Arm 2 (Tree-Only) Token Length Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Token Count")
plt.ylabel("Number of Samples")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


---
## Step 3. Qwen2.5-32B 베이스 모델 동결 & LoRA 어댑터 바인딩
### 💡 4-Arm 엄밀 통제 조건
* Arm 1과 100% 동일한 LoRA 하이퍼파라미터($r=16, \alpha=32$, 7개 선형 프로젝션 레이어)를 적용합니다.


In [ ]:
# Step 3 코드: Qwen2.5-32B 원본 동결 및 LoRA 어댑터 설정
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

print(f"[*] Qwen2.5-32B 가중치 로드 중 (bfloat16, SDPA, cuda:0): {model_dir}")
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    local_files_only=True,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="cuda:0"
)

model.config.use_cache = False
model.enable_input_require_grads()

model = get_peft_model(model, lora_config)
print("[+] LoRA 어댑터 바인딩 완료 (Arm 2 Tree-Only):")
model.print_trainable_parameters()


---
## Step 4. 실시간 학습 수렴 곡선 모니터링 콜백 (Live Progress Tracker)
### 💡 실시간 모니터링 포인트
* Arm 2(Tree-Only)는 정형화된 규칙 문장만을 학습하므로 초반 Train Loss가 매우 빠르게 하강할 수 있습니다.
* 그러나 검증 손실(Val Loss)이 안정적으로 유지되는지, 과적합 양상이 없는지 실시간으로 관찰합니다.


In [ ]:
# Step 4 코드: 실시간 손실 곡선 모니터링 콜백
from transformers import TrainerCallback

class RealtimePlotCallback(TrainerCallback):
    def __init__(self, plot_every=10):
        self.plot_every = plot_every
        self.steps = []
        self.train_losses = []
        self.val_steps = []
        self.val_losses = []
        self.lrs = []
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
            
        step = state.global_step
        if "loss" in logs:
            self.steps.append(step)
            self.train_losses.append(logs["loss"])
            if "learning_rate" in logs:
                self.lrs.append(logs["learning_rate"])
                
        if "eval_loss" in logs:
            self.val_steps.append(step)
            self.val_losses.append(logs["eval_loss"])
            
        if step % self.plot_every == 0 and len(self.train_losses) > 0:
            try:
                clear_output(wait=True)
                fig, ax1 = plt.subplots(figsize=(10, 4.2), dpi=130)
                
                ax1.plot(self.steps, self.train_losses, label="Train Loss (Arm 2 Tree-Only)", color="#8e44ad", lw=2)
                if self.val_losses:
                    ax1.plot(self.val_steps, self.val_losses, label="Val Loss", color="#e74c3c", marker="o", lw=2)
                ax1.set_xlabel("Global Steps", fontweight="bold")
                ax1.set_ylabel("Cross Entropy Loss", color="#8e44ad", fontweight="bold")
                ax1.tick_params(axis="y", labelcolor="#8e44ad")
                ax1.legend(loc="upper right", frameon=True)
                
                if self.lrs:
                    ax2 = ax1.twinx()
                    ax2.plot(self.steps[:len(self.lrs)], self.lrs, label="Learning Rate", color="#27ae60", linestyle=":", lw=1.5)
                    ax2.set_ylabel("Learning Rate", color="#27ae60", fontweight="bold")
                    ax2.tick_params(axis="y", labelcolor="#27ae60")
                    
                plt.title(f"Arm 2 (Tree-Only) Training Progress (Step: {step})", fontsize=12, fontweight="bold")
                fig.tight_layout()
                plt.show()
            except Exception:
                pass

print("[+] RealtimePlotCallback 준비 완료")


---
## Step 5. 표준 PyTorch Trainer 기반 Arm 2 LoRA 파인튜닝 가동
### 💡 훈련 하이퍼파라미터
* Arm 1과 완전히 동일한 배치 및 학습률 설정:
  `per_device_train_batch_size = 1`, `gradient_accumulation_steps = 16`, `learning_rate = 2e-4`, `num_train_epochs = 3`
* 가중치 저장 경로: `experiments/02_tree_ablation_xai/outputs/qwen2.5-32b-arm2-tree-only/`


In [ ]:
# Step 5 코드: Trainer 설정 및 실제 훈련 실행
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

arm2_output_dir = str(OUTPUTS_DIR / "qwen2.5-32b-arm2-tree-only")

training_args = TrainingArguments(
    output_dir=arm2_output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    callbacks=[RealtimePlotCallback(plot_every=5)]
)

print("=" * 70)
print("[*] Arm 2 (Tree-Only) 학습을 가동합니다...")
print(f" - 저장 위치: {arm2_output_dir}")
print("=" * 70)

trainer.train()

# 모델 및 토크나이저 저장
trainer.save_model(arm2_output_dir)
tokenizer.save_pretrained(arm2_output_dir)
print(f"[+] Arm 2 학습 완료! 어댑터 저장 완료: {arm2_output_dir}")


---
## Step 6. 최종 학습 완료 결과 요약 및 수렴 곡선 저장
### 💡 학습 지표 영구 기록
* 최종 수렴 곡선과 불확실성 감소 지표(Perplexity)를 `reports/arm2_training_loss_curve.png`로 저장합니다.


In [ ]:
# Step 6 코드: 최종 손실 수렴 및 Perplexity 시각화 저장
def plot_final_training_summary(train_history, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), dpi=150)
    
    # 1. Loss 수렴 곡선
    axes[0].plot(train_history.get("steps", []), train_history.get("train_loss", []), label="Train Loss", color="#8e44ad", lw=2)
    if "val_loss" in train_history:
        axes[0].plot(train_history.get("val_steps", []), train_history.get("val_loss", []), label="Val Loss", color="#e74c3c", marker="o", lw=2)
    axes[0].set_title("Arm 2 (Tree-Only) Loss Convergence", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Global Steps")
    axes[0].set_ylabel("Loss")
    axes[0].legend(frameon=True)
    
    # 2. Perplexity 곡선
    if "train_loss" in train_history:
        train_ppl = [math.exp(min(l, 10)) for l in train_history["train_loss"]]
        axes[1].plot(train_history.get("steps", []), train_ppl, label="Train Perplexity", color="#27ae60", lw=2)
        axes[1].set_title("Perplexity (Uncertainty)", fontsize=12, fontweight="bold")
        axes[1].set_xlabel("Global Steps")
        axes[1].set_ylabel("Perplexity (PPL)")
        axes[1].legend(frameon=True)
        
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    print(f"[+] 최종 학습 결과 그래프가 저장되었습니다: {save_path}")
    plt.show()

# 실제 로그 히스토리 추출
history = {"steps": [], "train_loss": [], "val_steps": [], "val_loss": []}
for log in trainer.state.log_history:
    if "loss" in log and "step" in log:
        history["steps"].append(log["step"])
        history["train_loss"].append(log["loss"])
    if "eval_loss" in log and "step" in log:
        history["val_steps"].append(log["step"])
        history["val_loss"].append(log["eval_loss"])

plot_final_training_summary(history, str(REPORTS_DIR / "arm2_training_loss_curve.png"))
